# Notebook 01: Chuẩn bị Dữ liệu Đa nguồn & Mô hình Giả lập Suy giảm OV2640 (Data Augmentation)
**Phụ trách:** Thành viên 2 (Thu thập & Trích xuất khung hình UAV) & Thành viên 4 (Mô hình suy giảm có tham chiếu)

### 1. Mục tiêu
1. Đảm bảo dữ liệu **đa nguồn (70+ chuỗi bay UAV khác nhau)** và áp dụng **Data Augmentation ngẫu nhiên** (xoay góc nghiêng roll $\pm 7^\circ$, phóng tỷ lệ độ cao, biến thiên mức độ ám hồng, góc mờ rung động $\theta \in [0^\circ, 180^\circ]$ và mức nén JPEG $Q \in [32, 58]$) để tránh hiện tượng **overfit bộ tham số**.
2. Kiểm tra tập dữ liệu gồm:
   - `data/raw_videos/`: 3 video bay FPV thực tế từ ESP32-CAM.
   - `data/real_uav/`: 60 ảnh kiểm thử UAV đã qua augment.
   - `data/synthetic_gt/` & `data/synthetic_degraded/`: 30 cặp ảnh đối chứng phục vụ tính **PSNR, SSIM, $\Delta E_{ab}$**.

In [1]:
import os, glob, cv2
import pandas as pd
import matplotlib.pyplot as plt

ROOT = "/home/ngtukien/Projects/XLA"
df_manifest = pd.read_csv(f"{ROOT}/data/dataset_manifest.csv")
print(f"Tổng số ảnh trong manifest: {len(df_manifest)}")
print(df_manifest["subset"].value_counts())
df_manifest.head(10)

Tổng số ảnh trong manifest: 90
real_uav_multisource_aug    40
synthetic_pair              30
real_uav_fpv_aug            20
Name: count, dtype: int64


,subset,filename,source_seq,rot_deg,scale,contrast_keep,hl_thresh,r_boost,b_boost,g_drop,motion_blur_len,motion_blur_angle,jpeg_quality
0,synthetic_pair,uav_esp32_sim_001.jpg,0000001,1.96,1.13,0.651,166.1,66.8,57.4,37.8,5,92.7,51
1,synthetic_pair,uav_esp32_sim_002.jpg,0000002,-0.95,1.19,0.693,172.9,62.8,35.5,44.5,3,28.4,52
2,synthetic_pair,uav_esp32_sim_003.jpg,0000003,6.00,1.09,0.729,166.0,54.2,36.2,22.7,5,148.4,38
3,synthetic_pair,uav_esp32_sim_004.jpg,0000006,-5.73,1.20,0.751,161.9,53.3,36.8,34.6,7,37.5,37
4,synthetic_pair,uav_esp32_sim_005.jpg,0000007,-6.93,1.19,0.629,147.7,64.8,48.4,42.2,3,93.7,41
5,synthetic_pair,uav_esp32_sim_006.jpg,0000008,6.76,1.20,0.736,162.0,43.6,42.2,34.2,3,160.7,34
6,synthetic_pair,uav_esp32_sim_007.jpg,0000010,-1.81,1.07,0.660,154.7,42.3,55.7,40.4,5,148.1,49
7,synthetic_pair,uav_esp32_sim_008.jpg,0000011,-3.00,1.20,0.752,166.3,38.3,29.8,30.5,5,133.7,41
8,synthetic_pair,uav_esp32_sim_009.jpg,0000013,1.64,1.09,0.770,145.0,64.8,36.5,45.5,5,16.4,32
9,synthetic_pair,uav_esp32_sim_010.jpg,0000016,-4.48,1.15,0.717,178.8,52.7,54.6,43.2,3,33.2,55
